In [ ]:
!pip install boto3 mne -q
!pip install scikit-optimize -q

import numpy as np
import os, gc
import mne
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import matplotlib.pyplot as plt

mne.set_log_level('ERROR')

# --- Top 20 by mu SNR (pre-screened from 215 subjects) ---
SELECTED = [
    'sub-032508', 'sub-032383', 'sub-032476', 'sub-032395', 'sub-032511',
    'sub-032361', 'sub-032502', 'sub-032469', 'sub-032394', 'sub-032324',
    'sub-032524', 'sub-032330', 'sub-032419', 'sub-032311', 'sub-032317',
    'sub-032370', 'sub-032355', 'sub-032332', 'sub-032396', 'sub-032526',
]

TARGET_FS = 1000.0
EPOCH_DURATION_S = 5.0
LOCAL_DIR = '/root/lemon_eeg'
os.makedirs(LOCAL_DIR, exist_ok=True)

HJORTH_CENTER = 'C3'
HJORTH_SURROUND = ['FC3', 'C1', 'CP3', 'C5']

BUCKET = 'fcp-indi'
PREFIX = 'data/Projects/INDI/MPI-LEMON/EEG_MPILMBB_LEMON/EEG_Raw_BIDS_ID/'

s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# ---------------------------------------------------------------
# Google Drive cache
# ---------------------------------------------------------------
DRIVE_BASE = '/content/drive/MyDrive/RAIN_IRC/LEMON'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"(Skipping Drive mount: {e})")
os.makedirs(DRIVE_BASE, exist_ok=True)

# Signature of the current preprocessing, stored alongside each cache file so a
# stale cache (different epoch length / fs / montage) is detected and rebuilt
# instead of silently loading mismatched epochs.
MONTAGE_SIG = f"{HJORTH_CENTER}-{'+'.join(HJORTH_SURROUND)}"
EXPECTED_LEN = int(EPOCH_DURATION_S * TARGET_FS)


def cache_path(subj_id):
    return os.path.join(DRIVE_BASE, f"{subj_id}.npz")


def save_subject(subj_id, sd):
    np.savez_compressed(
        cache_path(subj_id),
        epochs=sd['epochs'], fs=sd['fs'], n_epochs=sd['n_epochs'],
        cal_n=sd['cal_n'], test_n=sd['test_n'],
        montage=MONTAGE_SIG, epoch_len=EXPECTED_LEN,
    )


def load_subject(subj_id):
    """Return cached subject dict, or None if missing / stale / unreadable."""
    p = cache_path(subj_id)
    if not os.path.exists(p):
        return None
    try:
        d = np.load(p)
        if (d['epochs'].shape[1] != EXPECTED_LEN
                or float(d['fs']) != TARGET_FS
                or str(d['montage']) != MONTAGE_SIG):
            return None  # preprocessing changed -> treat as cache miss
        return {
            'epochs': d['epochs'], 'fs': float(d['fs']),
            'n_epochs': int(d['n_epochs']),
            'cal_n': int(d['cal_n']), 'test_n': int(d['test_n']),
        }
    except Exception:
        return None


def find_ch(ch_names, target):
    t = target.lower()
    for c in ch_names:
        if c.lower() == t: return c
    return None


def fix_vhdr(vhdr_path, subj_id):
    with open(vhdr_path, 'r') as f:
        lines = f.read().split('\n')
    for i, line in enumerate(lines):
        if line.startswith('DataFile='):
            lines[i] = f'DataFile={subj_id}.eeg'
        elif line.startswith('MarkerFile='):
            lines[i] = f'MarkerFile={subj_id}.vmrk'
    with open(vhdr_path, 'w') as f:
        f.write('\n'.join(lines))


print(f"Loading {len(SELECTED)} subjects (cache: {DRIVE_BASE})\n")

all_subjects = {}

for subj_id in SELECTED:
    # --- Try the Drive cache first ---
    cached = load_subject(subj_id)
    if cached is not None:
        all_subjects[subj_id] = cached
        print(f"  {subj_id}: cached ({cached['n_epochs']} epochs, "
              f"{cached['cal_n']} cal / {cached['test_n']} test)")
        continue

    # --- Cache miss: download + process, then persist ---
    subj_dir = os.path.join(LOCAL_DIR, subj_id)
    os.makedirs(subj_dir, exist_ok=True)

    subj_prefix = f"{PREFIX}{subj_id}/RSEEG/"
    try:
        resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=subj_prefix)
        keys = [f['Key'] for f in resp.get('Contents', [])]
    except:
        print(f"  {subj_id}: S3 error, skipping"); continue

    vhdr_path = None
    for key in keys:
        fname = key.split('/')[-1]
        local_path = os.path.join(subj_dir, fname)
        if not os.path.exists(local_path):
            try: s3.download_file(BUCKET, key, local_path)
            except: continue
        if fname.endswith('.vhdr'):
            vhdr_path = local_path

    if vhdr_path is None:
        print(f"  {subj_id}: no .vhdr, skipping"); continue

    fix_vhdr(vhdr_path, subj_id)

    try:
        raw = mne.io.read_raw_brainvision(vhdr_path, preload=True, verbose=False)
    except:
        print(f"  {subj_id}: MNE error, skipping"); continue

    center_ch = find_ch(raw.ch_names, HJORTH_CENTER)
    surround_chs = [find_ch(raw.ch_names, s) for s in HJORTH_SURROUND]
    if center_ch is None or None in surround_chs:
        print(f"  {subj_id}: missing channels, skipping"); del raw; continue

    raw.pick([center_ch] + surround_chs)
    if raw.info['sfreq'] != TARGET_FS:
        raw.resample(TARGET_FS, verbose=False)
    fs = raw.info['sfreq']

    center_data = raw.get_data(picks=[center_ch])[0]
    surround_data = raw.get_data(picks=surround_chs)
    hjorth = (center_data - surround_data.mean(axis=0)).astype(np.float32)
    del raw; gc.collect()

    epoch_samp = int(EPOCH_DURATION_S * fs)
    n_epochs = len(hjorth) // epoch_samp
    epochs = np.array([hjorth[i * epoch_samp:(i + 1) * epoch_samp]
                       for i in range(n_epochs)], dtype=np.float32)

    cal_n = int(0.6 * n_epochs)
    all_subjects[subj_id] = {
        'epochs': epochs, 'fs': fs, 'n_epochs': n_epochs,
        'cal_n': cal_n, 'test_n': n_epochs - cal_n,
    }
    save_subject(subj_id, all_subjects[subj_id])  # persist to Drive
    print(f"  {subj_id}: {n_epochs} epochs ({cal_n} cal / {n_epochs - cal_n} test) [saved]")

import shutil
shutil.rmtree(LOCAL_DIR, ignore_errors=True)

FS = TARGET_FS
n_loaded = len(all_subjects)
print(f"\nLoaded: {n_loaded} subjects @ {FS:.0f} Hz")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 54.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 8.0 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading 20 subjects (cache: /content/drive/MyDrive/RAIN_IRC/LEMON)

  sub-032508: 203 epochs (121 cal / 82 test) [saved]
  sub-032383: 208 epochs (124 cal / 84 test) [saved]
  sub-032476: 202 epochs (121 cal / 81 test) [saved]
  sub-032395: 215 epochs (129 cal / 86 test) [saved]
  sub-032511: 218 epochs (130 cal / 88 test) [saved]
  sub-032361: 201 epochs (120 cal / 81 test) [sa

/tmp/ipykernel_1511/460367159.py:145: RuntimeWarning: Missing header in marker file.
  raw = mne.io.read_raw_brainvision(vhdr_path, preload=True, verbose=False)


  sub-032419: 211 epochs (126 cal / 85 test) [saved]
  sub-032311: 202 epochs (121 cal / 81 test) [saved]
  sub-032317: 203 epochs (121 cal / 82 test) [saved]
  sub-032370: 200 epochs (120 cal / 80 test) [saved]
  sub-032355: 213 epochs (127 cal / 86 test) [saved]
  sub-032332: 200 epochs (120 cal / 80 test) [saved]
  sub-032396: 216 epochs (129 cal / 87 test) [saved]
  sub-032526: 202 epochs (121 cal / 81 test) [saved]

Loaded: 20 subjects @ 1000 Hz
